# 02d â€” Traditional ML: SVM, Random Forest, kNN

**Project:** UREP 32-0210-250078 | Crack Classification

**Pipeline:** Image -> Grayscale -> CLAHE -> 128x128 -> HOG+LBP+GLCM+Edge -> StandardScaler -> PCA(0.95) -> Classifier

No GPU / no PyTorch needed â€” entirely sklearn pipeline.

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import time
import random
import numpy as np
import joblib

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score

import config
from src.features import extract_features_from_directory
from src.evaluation import evaluate_predictions

# Reproducibility
random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "svm")
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

print(f"Model: Traditional ML (SVM, RF, kNN)")
print(f"Image size: {config.SVM_IMG_SIZE}x{config.SVM_IMG_SIZE} grayscale")
print(f"Output: {OUTPUT_DIR}")

## Feature Extraction

In [ ]:
t0 = time.time()
X_train, y_train, _ = extract_features_from_directory(config.SPLIT_DIR, subset="train", cache_dir=OUTPUT_DIR)
X_val, y_val, _ = extract_features_from_directory(config.SPLIT_DIR, subset="val", cache_dir=OUTPUT_DIR)
X_test, y_test, _ = extract_features_from_directory(config.SPLIT_DIR, subset="test", cache_dir=OUTPUT_DIR)
print(f"\nFeature extraction: {(time.time()-t0)/60:.1f} min")
print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## StandardScaler + PCA

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=config.SVM_PCA_VARIANCE, random_state=config.RANDOM_SEED)
X_train_p = pca.fit_transform(X_train_s)
X_val_p = pca.transform(X_val_s)
X_test_p = pca.transform(X_test_s)
print(f"PCA: {X_train_s.shape[1]} -> {X_train_p.shape[1]} components")

joblib.dump(scaler, os.path.join(OUTPUT_DIR, "models", "scaler.pkl"))
joblib.dump(pca, os.path.join(OUTPUT_DIR, "models", "pca.pkl"))

## SVM (RBF)

In [ ]:
t0 = time.time()
svm = SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced",
          probability=True, random_state=config.RANDOM_SEED)
svm.fit(X_train_p, y_train)
print(f"SVM training: {(time.time()-t0)/60:.1f} min")

val_f1 = f1_score(y_val, svm.predict(X_val_p), average="macro")
print(f"Val F1-macro: {val_f1:.4f}")
joblib.dump(svm, os.path.join(OUTPUT_DIR, "models", "best_svm.pkl"))

## Random Forest & kNN

In [ ]:
rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=config.RANDOM_SEED, n_jobs=-1)
rf.fit(X_train_p, y_train)
val_f1_rf = f1_score(y_val, rf.predict(X_val_p), average="macro")
print(f"RF Val F1-macro: {val_f1_rf:.4f}")
joblib.dump(rf, os.path.join(OUTPUT_DIR, "models", "random_forest.pkl"))

knn = KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1)
knn.fit(X_train_p, y_train)
val_f1_knn = f1_score(y_val, knn.predict(X_val_p), average="macro")
print(f"kNN Val F1-macro: {val_f1_knn:.4f}")
joblib.dump(knn, os.path.join(OUTPUT_DIR, "models", "knn.pkl"))

## Test Set Evaluation

In [ ]:
print("
SVM Test Evaluation:")
metrics_svm = evaluate_predictions(y_test, svm.predict(X_test_p), output_dir=OUTPUT_DIR, model_name="svm")

print("
Random Forest Test Evaluation:")
metrics_rf = evaluate_predictions(y_test, rf.predict(X_test_p), output_dir=OUTPUT_DIR, model_name="random_forest")

print("
kNN Test Evaluation:")
metrics_knn = evaluate_predictions(y_test, knn.predict(X_test_p), output_dir=OUTPUT_DIR, model_name="knn")

In [ ]:
print(f"\n{'='*65}")
print(f"TRADITIONAL ML â€” TEST SET COMPARISON")
print(f"{'='*65}")
print(f"  {'Model':<20} {'Accuracy':>10} {'F1-macro':>10} {'Mean IoU':>10}")
print(f"  {'-'*50}")
for name, m in [("SVM (RBF)", metrics_svm), ("Random Forest", metrics_rf), ("kNN (k=7)", metrics_knn)]:
    print(f"  {name:<20} {m['accuracy']:>10.4f} {m['f1_macro']:>10.4f} {m['mean_iou']:>10.4f}")
print(f"{'='*65}")